# Tutorial 1: Environments and Neural Networks

**zrth** extracts symbolic reactive modules from Python classes. You write a standard [Gymnasium](https://gymnasium.farama.org/) environment or a [PyTorch](https://pytorch.org/) neural network, then wrap it with `Wrapper()` or `Module()` — zrth analyzes it and produces a formal model with typed wires and terms that can be used for verification or training.

This tutorial walks through:
1. Defining an environment (plain `gym.Env`) and wrapping it with `zrth.gym.Wrapper`
2. Defining a neural network (plain `nn.Module`) and wrapping it with `zrth.torch.Module`
3. Inspecting the extracted modules
4. Training a ranking function to prove a liveness property
5. Formally verifying the ranking conditions with Z3

## Defining an environment

Write a standard [Gymnasium](https://gymnasium.farama.org/) environment by subclassing `gym.Env`:

- **`__init__`**: set `action_space`, `observation_space`, any parameters (e.g. `self.y0`), and state variables (e.g. `self.x`)
- **`reset`**: initialize all state variables, return `(observation, info)`
- **`step`**: update state given an action, return `(observation, reward, terminated, truncated, info)`

**Important:** state variables must also be assigned in `__init__` so the analyzer can infer their types. For example, even though `reset` sets `self.x = 0.0`, you should also write `self.x = 0.0` in `__init__`.

Below is a counter system with three variables $x$, $y$, $z$. The variable $x$ increments while $x < y \lor x < z$, and resets to $0$ otherwise. We want to show that $x = y \lor x = z$ must occur infinitely often.

In [1]:
import torch
import torch.nn as nn
import gymnasium as gym
from gymnasium import spaces
from zrth.gym import Wrapper
from zrth.torch import Module


class CounterEnv(gym.Env):
    """Counter environment: x increments while x < y or x < z, resets otherwise."""

    def __init__(self, y0, z0):
        super().__init__()

        self.action_space = spaces.Discrete(1)
        self.observation_space = spaces.Box(low=-1e6, high=1e6, shape=(1,))

        self.y0 = y0
        self.z0 = z0

        # Also set state variables here so the analyzer can infer their dtype
        self.x = 0.0
        self.y = y0
        self.z = z0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.x = 0.0
        self.y = self.y0
        self.z = self.z0

        observation = self.x
        return observation, {}

    def step(self, action):
        if self.x < self.y or self.x < self.z:
            self.x = self.x + 1.0
        else:
            self.x = 0.0

        # y and z remain constant
        self.y = self.y
        self.z = self.z

        at_target = self.x == self.y or self.x == self.z
        reward = 1.0 if at_target else 0.0
        terminated = at_target
        truncated = False
        observation = self.x
        return observation, reward, terminated, truncated, {}

## Wrapping and inspection

When you call `Wrapper(counter_instance)`, zrth analyzes the class's `__init__`, `reset`, and `step` methods. It:
1. Infers action/observation types from the gymnasium spaces
2. Classifies `self.*` attributes as **private** (read+write state) or **parameters** (read-only constants)
3. Creates typed wire pairs for each variable and signal
4. Converts `reset` into **init** terms and `step` into **update** terms

Print the extracted module to see the result.

In [2]:
counter = CounterEnv(y0=30.0, z0=50.0)
wrapped_counter = Wrapper(counter)
print(wrapped_counter)

module
  external
    w8, w9 : Float(1)
  interface
    w10, w11 : Float(1)
    w12, w13 : Float(1)
    w14, w15 : Bool(1)
    w16, w17 : Bool(1)
  private
    w0, w1 : Float(1)
    w2, w3 : Float(1)
    w4, w5 : Float(1)
  atom controls w1, w3, w5, w11, w13, w15, w17 reads w0, w2, w4
  init
    Tensor([50]) w6; 
    Tensor([30]) w7; 
    Tensor([0]) w18; 
    Id w5; w18
    Id w1; w7
    Id w3; w6
    Id w11; w18
    Tensor([0]) w13; 
    Const: false w15; 
    Const: false w17; 
  update
    Tensor([50]) w6; 
    Tensor([30]) w7; 
    Lt w19; w4, w0
    Lt w20; w4, w2
    Tensor([0]) w21; 
    Tensor([1]) w22; 
    Ite w23; w19, w22, w20
    Tensor([1]) w24; 
    Add w25; w4, w24
    Tensor([0]) w26; 
    Ite w27; w23, w25, w26
    Id w5; w27
    Id w1; w0
    Id w3; w2
    Eq w28; w27, w0
    Eq w29; w27, w2
    Tensor([0]) w30; 
    Tensor([1]) w31; 
    Ite w32; w28, w31, w29
    Tensor([1]) w33; 
    Tensor([0]) w34; 
    Ite w35; w32, w33, w34
    Tensor([0]) w36; 
    Id w11; w

## Reading the output

The printed module shows:

- **external**: input wires — here, the action (unused by this environment, but still present)
- **interface**: output wires — observation, reward, terminated, truncated
- **private**: internal state — wire pairs `wN, wM` for x, y, z (each pair is `latched, next`)
- **init**: terms from `reset` — how each wire is initialized
- **update**: terms from `step` — how each wire is updated, including `Ite` (if-then-else), `Lt`, `Eq`, `Add`

## Defining a neural network

Write a standard `nn.Module`, then wrap it with `Module(instance)` to extract a symbolic module:

- **`__init__`**: define `nn.Linear` layers
- **`forward`**: standard PyTorch forward pass

zrth extracts the architecture as a **combinatorial** (stateless) module. The resulting module has:
- **extl** (external input) wire pair — sized from the first layer's `in_features`
- **intf** (interface output) wire pair — sized from the last layer's `out_features`

The ranking function below maps $(x, y, z) \in \mathbb{R}^3$ to a scalar. It is used in formal verification to prove that the liveness property $x = y \lor x = z$ holds infinitely often — not a controller, so we don't compose it with the counter.

In [3]:
class RankingNN(nn.Module):
    """Ranking function: R(x, y, z) -> scalar.

    Architecture: [3] -> 2 -> [1]
    """

    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(3, 2)
        self.fc2 = nn.Linear(2, 1)

    def forward(self, state):
        x = torch.relu(self.fc1(state))
        return self.fc2(x)

Wrap and inspect. Note that the module has no `private` wires (it's stateless) and the `init`/`update` blocks are identical (combinatorial — same computation every step).

In [4]:
ranking = RankingNN()
wrapped_ranking = Module(ranking)
print(wrapped_ranking)

module
  external
    w37, w38 : Float(3)
  interface
    w39, w40 : Float(1)
  atom controls w40 awaits w38
  init
    Tensor([-0.14135563373565674 0.15505638718605042 0.40183013677597046 0.035399578511714935 -0.026347439736127853 ...]) w41; 
    Tensor([0.15030983090400696 0.3919077515602112]) w42; 
    Linear w43; w38, w41, w42
    ReLU w44; w43
    Tensor([-0.553821861743927 0.6773388981819153]) w45; 
    Tensor([0.41742128133773804]) w46; 
    Linear w47; w44, w45, w46
    Id w40; w47
  update
    Tensor([-0.14135563373565674 0.15505638718605042 0.40183013677597046 0.035399578511714935 -0.026347439736127853 ...]) w41; 
    Tensor([0.15030983090400696 0.3919077515602112]) w42; 
    Linear w43; w38, w41, w42
    ReLU w44; w43
    Tensor([-0.553821861743927 0.6773388981819153]) w45; 
    Tensor([0.41742128133773804]) w46; 
    Linear w47; w44, w45, w46
    Id w40; w47



## Training the ranking function

A **ranking function** $R: \mathcal{S} \to \mathbb{R}$ can be used to prove that a liveness property holds infinitely often. The key conditions are:

1. $R(s) \geq 0$ for all states $s$
2. $R(s') < R(s)$ whenever the property $x = y \lor x = z$ does *not* hold (the ranking strictly decreases)

If such an $R$ exists, the property must hold infinitely often — otherwise $R$ would decrease forever below zero, contradicting condition 1.

We train the `RankingNN` to approximate these conditions by:
1. Simulating the counter for $N$ steps (it's a fully functional `gym.Env`)
2. At each step, computing $R(x, y, z)$ and $R(x', y', z')$
3. Penalizing violations: $R$ should decrease at non-target states and be non-negative

In [5]:
optimizer = torch.optim.Adam(wrapped_ranking.parameters(), lr=0.01)
margin = 0.1
n_epochs = 100
n_steps = 20

# State variable names from the module
var_names = list(wrapped_counter.wire_names.keys())

for epoch in range(n_epochs):
    wrapped_counter.reset()

    # Collect trajectory of state dicts via wire_names
    states = []
    for _ in range(n_steps):
        state_dict = {v: getattr(wrapped_counter, v) for v in var_names}
        states.append(state_dict)
        wrapped_counter.step(0)

    loss = torch.tensor(0.0)
    for i in range(len(states) - 1):
        s = torch.tensor([states[i][v] for v in var_names], dtype=torch.float32)
        s_next = torch.tensor([states[i+1][v] for v in var_names], dtype=torch.float32)
        r = wrapped_ranking(s.unsqueeze(0)).squeeze()
        r_next = wrapped_ranking(s_next.unsqueeze(0)).squeeze()

        # Problem-specific: define when the liveness property holds
        at_target = (states[i]['x'] == states[i]['y']) or (states[i]['x'] == states[i]['z'])

        if not at_target:
            loss = loss + torch.relu(r_next - r + margin)
            loss = loss + torch.relu(-r)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 20 == 0:
        print(f"epoch {epoch:3d}  loss = {loss.item():.4f}")

epoch   0  loss = 68.5699
epoch  20  loss = 0.0000
epoch  40  loss = 0.0000
epoch  60  loss = 0.0000
epoch  80  loss = 0.0000


## Evaluating the trained ranking function

After training, we evaluate $R$ along a trajectory. At non-target states, $R$ should decrease on each step. These are approximate — formal verification follows below.

In [6]:
wrapped_counter.reset()

var_names = list(wrapped_counter.wire_names.keys())
header = "  ".join(f"{v:>5}" for v in var_names)
print(f"{'step':>4}  {header}  {'R(s)':>8}  {'target':>6}")
print("-" * (12 + 8 * len(var_names)))

with torch.no_grad():
    for step in range(102):
        state_dict = {v: getattr(wrapped_counter, v) for v in var_names}
        s = torch.tensor([state_dict[v] for v in var_names], dtype=torch.float32)
        r = wrapped_ranking(s.unsqueeze(0)).squeeze().item()

        # Problem-specific: target check
        at_target = (state_dict['x'] == state_dict['y']) or (state_dict['x'] == state_dict['z'])

        row = "  ".join(f"{state_dict[v]:5.1f}" for v in var_names)
        print(f"{step:4d}  {row}  {r:8.4f}  {'  *' if at_target else ''}")
        wrapped_counter.step(0)

step      y      z      x      R(s)  target
------------------------------------
   0   30.0   50.0    0.0   13.2570  
   1   30.0   50.0    1.0   13.0560  
   2   30.0   50.0    2.0   12.8549  
   3   30.0   50.0    3.0   12.6539  
   4   30.0   50.0    4.0   12.4528  
   5   30.0   50.0    5.0   12.2518  
   6   30.0   50.0    6.0   12.0507  
   7   30.0   50.0    7.0   11.8497  
   8   30.0   50.0    8.0   11.6486  
   9   30.0   50.0    9.0   11.4476  
  10   30.0   50.0   10.0   11.2465  
  11   30.0   50.0   11.0   11.0455  
  12   30.0   50.0   12.0   10.8444  
  13   30.0   50.0   13.0   10.6434  
  14   30.0   50.0   14.0   10.4423  
  15   30.0   50.0   15.0   10.2413  
  16   30.0   50.0   16.0   10.0402  
  17   30.0   50.0   17.0    9.8392  
  18   30.0   50.0   18.0    9.6381  
  19   30.0   50.0   19.0    9.4371  
  20   30.0   50.0   20.0    9.2360  
  21   30.0   50.0   21.0    9.0350  
  22   30.0   50.0   22.0    8.8339  
  23   30.0   50.0   23.0    8.6329  
  24   

## Formal verification with Z3

Training gives us a *candidate* ranking function, but empirical checks on a single trajectory cannot guarantee that the conditions hold for *every* reachable state.

We can use an SMT solver ([Z3](https://github.com/Z3Prover/z3)) to verify the ranking conditions exhaustively. The idea: **negate** a condition and ask Z3 whether a counterexample exists. If Z3 returns `unsat`, no counterexample exists and the condition holds universally.

`zrth.smt.z3` translates module terms into Z3 expressions — the same term IR that `zrth.eval` executes with PyTorch tensors.

### Building the symbolic state

First, create Z3 integer variables for the counter's state $(x, y, z)$ and execute the counter's update block symbolically. This gives us $x'$ — the next-state value of $x$ — as a Z3 expression.

In [7]:
import z3
from zrth.smt.z3 import z3_eval_itype

# Create Z3 variables for each private state wire (generic)
z3_vars = {}
state = {}
for name, (latched, nxt) in wrapped_counter.wire_names.items():
    z3_vars[name] = z3.Int(name)
    state[latched] = [z3_vars[name]]

# Execute the counter's update block term-by-term (generic)
for atom in wrapped_counter.atoms:
    for term in atom.update:
        read = [state[w] for w in term.read]
        results = z3_eval_itype(term.itype, read)
        for w, val in zip(term.write, results):
            state[w] = val

# Collect next-state variables (generic)
z3_next = {}
for name, (latched, nxt) in wrapped_counter.wire_names.items():
    z3_next[name] = state[nxt][0]

print("next-state expressions:")
for name, expr in z3_next.items():
    print(f"  {name}' = {expr}")

next-state expressions:
  y' = y
  z' = z
  x' = If(If(x < y, True, x < z), ToReal(x) + 1, 0)


### Building the symbolic ranking function

Execute the ranking module's terms with Z3 to get $R(s)$ and $R(s')$ as Z3 expressions. The trained weights become exact rational constants in Z3.

In [8]:
extl = list(wrapped_ranking.extl)
intf = list(wrapped_ranking.intf)

def eval_ranking(input_vars):
    """Evaluate the ranking module's terms with Z3, returning R as a Z3 expression."""
    r_state = {extl[0][1]: input_vars}
    for atom in wrapped_ranking.atoms:
        for term in atom.update:
            read = [r_state[w] for w in term.read]
            results = z3_eval_itype(term.itype, read)
            for w, val in zip(term.write, results):
                r_state[w] = val
    return r_state[intf[0][1]][0]

# Build Z3 variable lists in wire_names order (generic)
current_vars = [z3_vars[name] for name in wrapped_counter.wire_names]
next_vars = [z3_next[name] for name in wrapped_counter.wire_names]

# R(s) and R(s') (generic)
R_s = eval_ranking(current_vars)
R_s_next = eval_ranking(next_vars)

# Problem-specific: domain constraints and target predicate
x, y, z = z3_vars['x'], z3_vars['y'], z3_vars['z']
domain = [y == 30, z == 50, z3.And(x >= 0, z3.Or(x < y, x < z))]

print("R(s) and R(s') built as Z3 expressions")

R(s) and R(s') built as Z3 expressions


### Verifying the conditions

We negate each condition and check satisfiability. If Z3 returns `unsat`, the condition holds for all reachable states. If `sat`, the model is a concrete counterexample.

**Condition 1:** $R(s) \geq 0$ for all states

In [9]:
# Condition 1: R(s) >= 0 for all reachable states
solver = z3.Solver()
solver.add(*domain)
solver.add(R_s < 0)          # negate: R(s) < 0

result = solver.check()
if result == z3.unsat:
    print("VERIFIED: R(s) >= 0 for all reachable states")
else:
    m = solver.model()
    vals = {name: m[z3_vars[name]] for name in z3_vars}
    print(f"COUNTEREXAMPLE (condition 1): {vals}")
    print(f"  R(s) = {float(m.eval(R_s, model_completion=True).as_fraction()):.6f} (should be >= 0)")

VERIFIED: R(s) >= 0 for all reachable states


**Condition 2:** $R(s') < R(s)$ when not at target (strict decrease)

In [10]:
# Condition 2: R(s') < R(s) - margin/2 when not at target
solver = z3.Solver()
solver.add(*domain)
solver.add(R_s_next >= R_s - margin/2)     # negate: R does not decrease by margin/2

result = solver.check()
if result == z3.unsat:
    print("VERIFIED: R strictly decreases at all non-target states")
else:
    m = solver.model()
    vals = {name: m[z3_vars[name]] for name in z3_vars}
    print(f"COUNTEREXAMPLE (condition 2): {vals}")
    r_val = float(m.eval(R_s, model_completion=True).as_fraction())
    r_next_val = float(m.eval(R_s_next, model_completion=True).as_fraction())
    print(f"  R(s) = {r_val:.6f}, R(s') = {r_next_val:.6f} (should be R(s') < R(s))")

VERIFIED: R strictly decreases at all non-target states


## Interactive module viewer

Finally, launch the interactive viewer to explore the counter module's wires and terms visually.

In [11]:
from zrth.visual import show
stop = show(wrapped_counter)

Module viewer running at http://127.0.0.1:58410 (ws://127.0.0.1:58411)
